## Tutorial for CellGRN and CellGRN-sparse

This tutorial demonstrates CellGRN on a 500-cell multiome dataset and then runs CellGRN-sparse on the same input. CellGRN outputs cell-specific, sample-wise, and cell-type-wise GRNs. CellGRN-sparse outputs sample-wise and cell-type-wise GRNs only; it does not store the cell-specific GRN matrix.

Datasets can be downloaded from the Figshare repository at https://doi.org/10.6084/m9.figshare.31304758. After uncompressing the data, update the path variables in the next cell to match your local directory layout.

The GRN backbone table must contain these columns:

| group_subtype | feature1 | feature2 |
| --- | --- | --- |
| TF-Gene | NFKB1 | IL6 |
| TF-Peak | NFKB1 | chr1:1000-1500 |
| Peak-Gene | chr1:1000-1500 | IL6 |

Formatted outputs contain `TF`, `Gene`, `Peak`, `Score`, and, for cell-type-wise files, `cell_type`. By default, `format_sample_grn` writes the top-ranked edges rather than all edges: top 10,000 TF-gene, top 20,000 TF-peak, and top 20,000 peak-gene edges.

In [ ]:
# 1. Load required packages
from pathlib import Path

from cellgrn.main import (
    SparseGRNCalculator,
    compute_all_cells_grn,
    format_celltype_grn,
    format_sample_grn,
    normalize_rna,
    normalize_rna_sparse,
    parse_edges,
    summarize_grn,
)

import gc
import os
import pickle

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

In [ ]:
# 2. Set paths and load the LINGER backbone used in this example
# Change these two paths after downloading and uncompressing the data.
BACKBONE_DIR = Path("/path/to/uncompressed/cellGRN_data/scalability")
MULTIOME_DIR = Path("/path/to/uncompressed/benchmark/bench_dataset/scalability/c500")

PROJECT_ROOT = Path.cwd()
TF_FILE = PROJECT_ROOT / "all_hg_TF.txt"
OUTPUT_DIR = PROJECT_ROOT / "output"
SPARSE_OUTPUT_DIR = PROJECT_ROOT / "output2"

cand_df = pd.read_csv(BACKBONE_DIR / "linger_grn.csv", header=0)

input_genes = [i.rstrip() for i in open(BACKBONE_DIR / "linger_gene.txt")]
input_peaks = [i.rstrip() for i in open(BACKBONE_DIR / "linger_peak.txt")]
all_tf = [i.rstrip() for i in open(TF_FILE)]

In [ ]:
# 3. Load a 500-cell multiome dataset
cell_meta = pd.read_csv(MULTIOME_DIR / "metadata.csv", index_col=0)

input_rna = ad.read_h5ad(MULTIOME_DIR / "BMMC-multiome-c500-RNA-counts.h5ad")
input_atac = ad.read_h5ad(MULTIOME_DIR / "BMMC-multiome-c500-ATAC-peaks.h5ad")

# Align metadata to the h5ad cell order before summarizing by cell type.
cell_meta = cell_meta.loc[input_rna.obs_names].copy()
cell_types = cell_meta["cell_type.l1"]

input_gene = input_rna.var.index.values
input_peak = input_atac.var.index.values
input_tf = list(set(input_gene) & set(all_tf))

rna_counts = input_rna.X.toarray() if sparse.issparse(input_rna.X) else input_rna.X
atac_counts = input_atac.X.toarray() if sparse.issparse(input_atac.X) else input_atac.X

input_df1 = pd.DataFrame(rna_counts, index=input_rna.obs.index.values, columns=input_rna.var.index.values)
peak_rename = [i.replace("-", ":", 1) for i in input_atac.var.index.values]
input_df2 = pd.DataFrame(atac_counts, index=input_atac.obs.index.values, columns=peak_rename)

input_df1 = input_df1[input_genes]
input_df2 = input_df2[input_peaks]

rna_data1, rna_data2 = normalize_rna(input_df1)
atac_data = input_df2.copy()

input_tfs = [tf for tf in input_tf if tf in input_genes]
tf_data1 = rna_data1[input_tfs].copy()
tf_data2 = rna_data2[input_tfs].copy()

In [ ]:
# 4. Run CellGRN
edges_idx, edges_name = parse_edges(cand_df, input_tfs, input_genes, input_peaks)

grn_scale2 = compute_all_cells_grn(
    tf_data2,
    rna_data2,
    atac_data,
    edges_idx,
    edges_name,
    input_tfs,
    input_genes,
    input_peaks,
)

sample_grn_scale2, celltype_grn_scale2 = summarize_grn(grn_scale2, cell_types)

tf_gene_res_scale2, tf_peak_res_scale2, gene_peak_res_scale2 = format_sample_grn(sample_grn_scale2)
tf_gene_ct_res_scale2, tf_peak_ct_res_scale2, gene_peak_ct_res_scale2 = format_celltype_grn(celltype_grn_scale2)

In [ ]:
# 5. Save cell-specific, cell-type-wise, and sample-wise GRN results
# The cell-specific GRN matrix is saved as a pickle file.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "demo_linger_cell_grn.pkl", "wb") as f:
    pickle.dump(grn_scale2.copy(), f)

tf_gene_res_scale2.to_csv(OUTPUT_DIR / "tf_gene_sample_scale2.csv", index=False)
tf_peak_res_scale2.to_csv(OUTPUT_DIR / "tf_peak_sample_scale2.csv", index=False)
gene_peak_res_scale2.to_csv(OUTPUT_DIR / "gene_peak_sample_scale2.csv", index=False)

tf_gene_ct_res_scale2.to_csv(OUTPUT_DIR / "tf_gene_celltype_scale2.csv", index=False)
tf_peak_ct_res_scale2.to_csv(OUTPUT_DIR / "tf_peak_celltype_scale2.csv", index=False)
gene_peak_ct_res_scale2.to_csv(OUTPUT_DIR / "gene_peak_celltype_scale2.csv", index=False)

In [ ]:
# 6. Optional: run CellGRN-sparse
# CellGRN-sparse returns sample-wise and cell-type-wise GRNs only.
input_rna = ad.read_h5ad(MULTIOME_DIR / "BMMC-multiome-c500-RNA-counts.h5ad")
input_atac = ad.read_h5ad(MULTIOME_DIR / "BMMC-multiome-c500-ATAC-peaks.h5ad")
cell_types = cell_meta.loc[input_rna.obs_names, "cell_type.l1"].to_numpy()

peak_rename = [i.replace("-", ":", 1) for i in input_atac.var.index.values]
input_atac.var.index = peak_rename

gene_indices = [input_rna.var_names.get_loc(g) for g in input_genes if g in input_rna.var_names]
peak_indices = [input_atac.var_names.get_loc(p) for p in input_peaks if p in input_atac.var_names]

rna_sub = input_rna[:, gene_indices].copy()
atac_sub = input_atac[:, peak_indices].copy()

current_genes = rna_sub.var.index.values
current_peaks = atac_sub.var.index.values

input_tfs = [tf for tf in current_genes if tf in all_tf]
tf_indices_in_sub = [rna_sub.var_names.get_loc(tf) for tf in input_tfs]

rna_data1_sparse, rna_data2_dense = normalize_rna_sparse(rna_sub.X)
tf_data2_arr = rna_data2_dense[:, tf_indices_in_sub]

edges_idx, edges_name = parse_edges(cand_df, input_tfs, current_genes, current_peaks)
calculator = SparseGRNCalculator(
    edges_idx,
    edges_name,
    input_tfs,
    current_genes,
    current_peaks,
)

batch_size = 1000
n_total = rna_sub.shape[0]
atac_sparse = atac_sub.X

for i in range(0, n_total, batch_size):
    end_i = min(i + batch_size, n_total)

    batch_tf = tf_data2_arr[i:end_i]
    batch_rna = rna_data2_dense[i:end_i]
    batch_atac = atac_sparse[i:end_i]
    batch_ct = cell_types[i:end_i]

    calculator.process_batch(batch_tf, batch_rna, batch_atac, batch_ct)

    if i % 5000 == 0:
        gc.collect()

sample_grn_scale2, celltype_grn_scale2 = calculator.finalize()

In [ ]:
tf_gene_res_scale2, tf_peak_res_scale2, gene_peak_res_scale2 = format_sample_grn(sample_grn_scale2)
tf_gene_ct_res_scale2, tf_peak_ct_res_scale2, gene_peak_ct_res_scale2 = format_celltype_grn(celltype_grn_scale2)

SPARSE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tf_gene_res_scale2.to_csv(SPARSE_OUTPUT_DIR / "tf_gene_sample_scale2.csv", index=False)
tf_peak_res_scale2.to_csv(SPARSE_OUTPUT_DIR / "tf_peak_sample_scale2.csv", index=False)
gene_peak_res_scale2.to_csv(SPARSE_OUTPUT_DIR / "gene_peak_sample_scale2.csv", index=False)

tf_gene_ct_res_scale2.to_csv(SPARSE_OUTPUT_DIR / "tf_gene_celltype_scale2.csv", index=False)
tf_peak_ct_res_scale2.to_csv(SPARSE_OUTPUT_DIR / "tf_peak_celltype_scale2.csv", index=False)
gene_peak_ct_res_scale2.to_csv(SPARSE_OUTPUT_DIR / "gene_peak_celltype_scale2.csv", index=False)